In [1]:
import common as cm
import pandas as pd
import os
import numpy as np

In [2]:
# Load near-discrete PRS models once (mode share > 0.5: more than half of individuals share one score)
qc_df = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "prs_model_qc_eur.csv"))
flagged = set(zip(qc_df.loc[qc_df["flag_mode_0.5"], "prs_phenocode"], qc_df.loc[qc_df["flag_mode_0.5"], "model_file"]))

records = []

for icd in cm.phecode2icd["icd"]:
    train_path = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_training", f"{icd}.csv")
    test_path = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_testing", f"{icd}.csv")

    # Skip traits missing either ranking file
    if not (os.path.exists(train_path) and os.path.exists(test_path)):
        continue

    # Read ranking files
    # Columns: auc, prs_phenocode, model_file, n_samples, n_cases
    train_rank = pd.read_csv(train_path)
    test_rank = pd.read_csv(test_path)

    # Flag near-discrete models in both halves (QC is defined on the full EUR score distribution)
    is_flagged_train = pd.Series([(p, m) in flagged for p, m in zip(train_rank["prs_phenocode"], train_rank["model_file"])], index=train_rank.index)
    is_flagged_test = pd.Series([(p, m) in flagged for p, m in zip(test_rank["prs_phenocode"], test_rank["model_file"])], index=test_rank.index)

    # Set their AUC to 0.5 so they can neither be selected nor count as the oracle
    train_rank.loc[is_flagged_train, "auc"] = 0.5
    test_rank.loc[is_flagged_test, "auc"] = 0.5

    # Re-sort by AUC descending and reset the index so row 0 / rank positions are correct
    train_rank = train_rank.sort_values("auc", ascending=False).reset_index(drop=True)
    test_rank = test_rank.sort_values("auc", ascending=False).reset_index(drop=True)

    # Select the best single PRS on the training half (row 0)
    train_best = train_rank.iloc[0]
    best_model = train_best["model_file"]

    # Locate the same PRS model in the testing half
    # Match on model_file because one phenocode has three PRS methods
    test_match = test_rank.loc[test_rank["model_file"] == best_model]
    test_row = test_match.iloc[0]

    # Testing-half rank of the train-selected PRS (1 = best)
    test_rank_pos = test_match.index[0] + 1

    # Oracle best single PRS: selected directly on the testing half (optimistic upper bound)
    oracle_row = test_rank.iloc[0]

    # Map the selected PRS's phenocode back to its ICD code
    prs_phecode = str(train_best["prs_phenocode"])
    prs_icd = cm.phecode2icd.get(prs_phecode, np.nan)

    records.append({
        "icd": icd,
        "auc": test_row["auc"],                    # train-selected PRS, evaluated on testing half
        "auc_train": train_best["auc"],            # same PRS, AUC on training half
        "test_rank": test_rank_pos,                # its rank among all PRS on testing half
        "prs_phecode": prs_phecode,
        "prs_icd": prs_icd,
        "auc_oracle": oracle_row["auc"],           # oracle best single AUC on testing half
        "n_case": test_row["n_cases"],
        "n_sample": test_row["n_samples"]
    })

# Assemble and save the best-single summary table
df = pd.DataFrame(records)
report_file = os.path.join(cm.ROOT_PATH, "result", "ensembleprs", "aou_bestsingle_result.csv")
df.to_csv(report_file, index=False)

In [3]:
# Load single-SNP C+T phenocodes once (a single variant is not a polygenic score)
qc_df = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "prs_model_qc_eur.csv"))
flagged = set(zip(qc_df.loc[qc_df["flag_mode_0.5"], "prs_phenocode"], qc_df.loc[qc_df["flag_mode_0.5"], "model_file"]))

records = []

for icd in cm.phecode2icd["icd"]:
    train_path = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_training", f"{icd}.csv")
    ukb_path = os.path.join(cm.ROOT_PATH, "prs_ranking", "UKB", f"{icd}.csv")

    # Skip traits missing either ranking file
    if not (os.path.exists(train_path) and os.path.exists(ukb_path)):
        continue

    # Read ranking files
    # Columns: auc, prs_phenocode, model_file, n_samples, n_cases
    train_rank = pd.read_csv(train_path)
    ukb_rank = pd.read_csv(ukb_path)

    # Flag single-SNP C+T scores in both cohorts (method is encoded in model_file, e.g. "_C+T-pseudo")
    is_flagged_train = pd.Series([(p, m) in flagged for p, m in zip(train_rank["prs_phenocode"], train_rank["model_file"])], index=train_rank.index)
    is_flagged_test = pd.Series([(p, m) in flagged for p, m in zip(test_rank["prs_phenocode"], test_rank["model_file"])], index=test_rank.index)

    # Set their AUC to 0.5 so they can neither be selected nor count as the oracle
    train_rank.loc[is_flagged_train, "auc"] = 0.5
    test_rank.loc[is_flagged_test, "auc"] = 0.5

    # Re-sort by AUC descending and reset the index so row 0 / rank positions are correct
    train_rank = train_rank.sort_values("auc", ascending=False).reset_index(drop=True)
    ukb_rank = ukb_rank.sort_values("auc", ascending=False).reset_index(drop=True)

    # Select the best single PRS on the AoU training half (row 0)
    train_best = train_rank.iloc[0]
    best_model = train_best["model_file"]

    # Locate the same PRS model in UKB
    # Match on model_file because one phenocode has three PRS methods
    ukb_match = ukb_rank.loc[ukb_rank["model_file"] == best_model]
    if ukb_match.empty:
        continue
    ukb_row = ukb_match.iloc[0]

    # UKB rank of the AoU-train-selected PRS (1 = best)
    ukb_rank_pos = ukb_match.index[0] + 1

    # Oracle best single PRS: selected directly in UKB (optimistic upper bound)
    oracle_row = ukb_rank.iloc[0]

    # Map the selected PRS's phenocode back to its ICD code
    prs_phecode = str(train_best["prs_phenocode"])
    prs_icd = cm.phecode2icd.get(prs_phecode, np.nan)

    records.append({
        "icd": icd,
        "auc": ukb_row["auc"],                     # AoU-train-selected PRS, evaluated in UKB
        "auc_train": train_best["auc"],            # same PRS, AUC on AoU training half
        "ukb_rank": ukb_rank_pos,                  # its rank among all PRS in UKB
        "prs_phecode": prs_phecode,
        "prs_icd": prs_icd,
        "auc_oracle": oracle_row["auc"],           # oracle best single AUC in UKB
        "n_case": ukb_row["n_cases"],
        "n_sample": ukb_row["n_samples"]
    })

# Assemble and save the best-single summary table
df = pd.DataFrame(records)
report_file = os.path.join(cm.ROOT_PATH, "result", "ensembleprs", "ukb_bestsingle_result.csv")
df.to_csv(report_file, index=False)